# **All nessary inputs**

In [ ]:
!pip install boto3 pandas scikit-learn tensorflow

import boto3
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras import Input
from tensorflow.keras.layers import LSTM, Dense, Dropout, Conv1D, MaxPooling1D, Flatten
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

# **DATA loading**

In [ ]:
import os
import logging
import pandas as pd
import json
# Initialize S3 client
s3 = boto3.client('s3',
                  region_name='us-east-1'  # Replace with your bucket's region
                  )

# Specify your bucket name
bucket_name = 'iot-sensor-data'

# Initialize an empty list to store the data
all_data = []
continuation_token = None

# Pagination loop to retrieve all objects
while True:
    if continuation_token:
        response = s3.list_objects_v2(Bucket=bucket_name, ContinuationToken=continuation_token)
    else:
        response = s3.list_objects_v2(Bucket=bucket_name)

    # Check if 'Contents' key exists in the response to avoid errors
    if 'Contents' in response:
        for obj in response['Contents']:
            key = obj['Key']  # Get the file name (key) from S3

            # Check if the file is a JSON file
            if key.endswith('.json'):
                # Download the JSON file
                file_obj = s3.get_object(Bucket=bucket_name, Key=key)

                try:
                    file_content = json.loads(file_obj['Body'].read().decode('utf-8'))
                    all_data.append(file_content)  # Only append if successful
                except json.JSONDecodeError as e:
                    print(f"Error decoding JSON in file {key}: {e}")
                except Exception as e:
                    print(f"Error reading file {key}: {e}")

    # Check if there are more objects to retrieve
    if response.get('IsTruncated'):  # True if there are more objects to fetch
        continuation_token = response['NextContinuationToken']
    else:
        break

# Create a pandas DataFrame from the list of dictionaries
if all_data:
    df = pd.DataFrame(all_data)
    print("Data successfully loaded into DataFrame:")
    print(df)

     # Checking for null values in each column
    for column in df.columns:
        null_count = df[column].isnull().sum()
        print(f"{column} NULL value: {null_count}")

else:
    print("No objects found in the bucket.")


# **Correlation Heat Map of Raw Data**

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Calculate the correlation matrix for the smoothed data
correlation_matrix = df[['temperature', 'cpu_usage', 'memory_usage', 'network_bandwidth']].corr()

# Display the correlation matrix
print("Correlation Matrix:")
print(correlation_matrix)

# Plot the correlation matrix as a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title("Correlation Matrix Heatmap")
plt.show()


# **NUll value counting**
 //check if the vaules are null    

In [ ]:
# Calculate the percentage of null values for each column
null_percentage = (df.isnull().sum() / len(df)) * 100

# Plot the null percentages as a bar chart
plt.figure(figsize=(10, 6))
null_percentage.plot(kind='bar', color='salmon')
plt.title('Percentage of Null Values per Column')
plt.xlabel('Columns')
plt.ylabel('Percentage of Null Values (%)')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


# **Making a copy of the data and then Erasing noise using savgol filter**

In [ ]:
from scipy.signal import savgol_filter
# Make a copy of the original DataFrame for smoothing
df_smoothed = df.copy()

# Columns to apply smoothing (replace with your actual columns)
columns_to_smooth = ['temperature', 'cpu_usage', 'memory_usage', 'network_bandwidth']

# Apply Savitzky-Golay filter to each column in the copy
for col in columns_to_smooth:
    df_smoothed[col] = savgol_filter(df_smoothed[col].values, 121, 3, mode='nearest')

# Function to visualize original and smoothed data
def show_comparison_plot(column):
    plt.figure(figsize=(10, 6))
    plt.plot(df.index, df[column], label='Original', color='red', linestyle='--', linewidth=0.7)
    plt.plot(df_smoothed.index, df_smoothed[column], label='Smoothed', color='blue', linewidth=0.7)
    plt.title(f"Comparison of Original and Smoothed Data for {column}")
    plt.xlabel("Time (Min)")
    plt.ylabel(column)
    plt.legend()
    plt.grid(True)
    plt.show()

# Example usage: plot comparison for each column
for col in columns_to_smooth:
    show_comparison_plot(col)

# **Saving the data to s3 bucket**
// as CSV

In [ ]:
import boto3
import pandas as pd
import json

# Save to a temporary CSV file
temp_file = '/tmp/smoothed_data.csv'  # For local use; change path if needed
df_smoothed.to_csv(temp_file, index=False)

# Upload the CSV file to S3
s3 = boto3.client('s3',
                  region_name='us-east-1'  # Replace with your bucket's region
                  )
bucket_name = 'iot-sensor-data'
s3_file_key = 'smoothed_data/smoothed_data.csv'  # Adjust the path as needed in S3

# Upload the file to S3
s3.upload_file(temp_file, bucket_name, s3_file_key)
print("Smoothed data has been saved to S3.")


# **Heatmap of correlation  of pre proccesed data**

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Calculate the correlation matrix for the smoothed data
correlation_matrix = df_smoothed[['temperature', 'cpu_usage', 'memory_usage', 'network_bandwidth']].corr()

# Display the correlation matrix
print("Correlation Matrix:")
print(correlation_matrix)

# Plot the correlation matrix as a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title("Correlation Matrix Heatmap")
plt.show()


# **Graphical representation of each metrics after noise reduction**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import gridspec
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import boto3
import io

# Initialize S3 client
s3 = boto3.client('s3',
                  region_name='us-east-1'  # Replace with your bucket's region
                  )

# Define S3 bucket and file key
bucket_name = 'iot-sensor-data'
file_key = 'smoothed_data/smoothed_data.csv'  # Replace with the actual path to your CSV file in S3

# Load data from S3 into a pandas DataFrame
file_obj = s3.get_object(Bucket=bucket_name, Key=file_key)
file_content = file_obj['Body'].read().decode('utf-8')
dataset = pd.read_csv(io.StringIO(file_content))

# Select relevant columns for analysis
columns_to_analyze = [ 'cpu_usage', 'network_bandwidth', 'temperature', 'memory_usage']  # Adjust as needed

# Define function to plot time-series, ACF, and PACF
def ts_plot(data, column_name):
    fig = plt.figure(figsize=(14, 10))
    gs = gridspec.GridSpec(2, 2)

    # Plot the time-series
    ax_ts = fig.add_subplot(gs[0, 0:2])
    ax_acf = fig.add_subplot(gs[1, 0])
    ax_pacf = fig.add_subplot(gs[1, 1])

    ax_ts.plot(range(len(data)), data)
    ax_ts.set_title(f"Time Series Plot of {column_name}")

    # Plot ACF and PACF
    plot_acf(data, ax=ax_acf, lags=30)
    ax_acf.set_title(f"Autocorrelation Function (ACF) of {column_name}")

    plot_pacf(data, ax=ax_pacf, lags=30)
    ax_pacf.set_title(f"Partial Autocorrelation Function (PACF) of {column_name}")

    plt.tight_layout()
    plt.show()

# Generate plots for each selected column
for column in columns_to_analyze:
    if column in dataset.columns:  # Check if the column exists in the dataset
        data = dataset[column].dropna().values  # Drop NA values and get values for analysis
        ts_plot(data, column)
    else:
        print(f"Column '{column}' not found in the dataset.")


# **Comparing it with the Raw data**

In [ ]:
import os
import logging
import pandas as pd
import json
import boto3
import matplotlib.pyplot as plt
from matplotlib import gridspec
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Initialize S3 client
s3 = boto3.client('s3',
                  region_name='us-east-1'  # Replace with your bucket's region
                  )

# Specify your bucket name
bucket_name = 'iot-sensor-data'

# Initialize an empty list to store the data
all_data = []
continuation_token = None

# Pagination loop to retrieve all objects
while True:
    if continuation_token:
        response = s3.list_objects_v2(Bucket=bucket_name, ContinuationToken=continuation_token)
    else:
        response = s3.list_objects_v2(Bucket=bucket_name)

    # Check if 'Contents' key exists in the response to avoid errors
    if 'Contents' in response:
        for obj in response['Contents']:
            key = obj['Key']  # Get the file name (key) from S3

            # Check if the file is a JSON file
            if key.endswith('.json'):
                # Download the JSON file
                file_obj = s3.get_object(Bucket=bucket_name, Key=key)

                try:
                    file_content = json.loads(file_obj['Body'].read().decode('utf-8'))
                    all_data.append(file_content)  # Only append if successful
                except json.JSONDecodeError as e:
                    print(f"Error decoding JSON in file {key}: {e}")
                except Exception as e:
                    print(f"Error reading file {key}: {e}")

    # Check if there are more objects to retrieve
    if response.get('IsTruncated'):  # True if there are more objects to fetch
        continuation_token = response['NextContinuationToken']
    else:
        break

# Convert the list of dictionaries into a DataFrame
if all_data:
    dataset = pd.DataFrame(all_data)
else:
    print("No data found in S3 bucket.")
    dataset = pd.DataFrame()  # Empty DataFrame if no data

# Select relevant columns for analysis
columns_to_analyze = ['cpu_usage', 'network_bandwidth', 'temperature', 'memory_usage']  # Adjust as needed

# Define function to plot time-series, ACF, and PACF
def ts_plot(data, column_name):
    fig = plt.figure(figsize=(14, 10))
    gs = gridspec.GridSpec(2, 2)

    # Plot the time-series
    ax_ts = fig.add_subplot(gs[0, 0:2])
    ax_acf = fig.add_subplot(gs[1, 0])
    ax_pacf = fig.add_subplot(gs[1, 1])

    ax_ts.plot(range(len(data)), data)
    ax_ts.set_title(f"Time Series Plot of {column_name}")

    # Plot ACF and PACF
    plot_acf(data, ax=ax_acf, lags=30)
    ax_acf.set_title(f"Autocorrelation Function (ACF) of {column_name}")

    plot_pacf(data, ax=ax_pacf, lags=30)
    ax_pacf.set_title(f"Partial Autocorrelation Function (PACF) of {column_name}")

    plt.tight_layout()
    plt.show()

# Check if there is data in the DataFrame
if not dataset.empty:
    # Generate plots for each selected column
    for column in columns_to_analyze:
        if column in dataset.columns:  # Check if the column exists in the dataset
            data = dataset[column].dropna().values  # Drop NA values and get values for analysis
            ts_plot(data, column)
        else:
            print(f"Column '{column}' not found in the dataset.")
else:
    print("Dataset is empty, no plots will be generated.")


# **IMPORT DATA From CSV file**

In [ ]:
import boto3
import pandas as pd
import io

# Initialize S3 client with your AWS credentials
# Upload the CSV file to S3
s3 = boto3.client('s3',
                  region_name='us-east-1'  # Replace with your bucket's region
                  )
bucket_name = 'iot-sensor-data'
s3_file_key = 'smoothed_data/smoothed_data.csv'   # Ensure this matches the path in your S3 bucket

# Load the CSV data from S3 into a pandas DataFrame
try:
    file_obj = s3.get_object(Bucket=bucket_name, Key=file_key)
    file_content = file_obj['Body'].read().decode('utf-8')
    smoothed_data = pd.read_csv(io.StringIO(file_content))
    print("Data loaded successfully.")
    print(smoothed_data)  # Display the first few rows
except Exception as e:
    print(f"Error loading data from S3: {e}")


# **LSTM Model**

In [ ]:
import boto3
import pandas as pd
import io
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Initialize S3 client with your AWS credentials
s3 = boto3.client('s3',
                region_name='us-east-1')

bucket_name = 'iot-sensor-data'
s3_file_key = 'smoothed_data/smoothed_data.csv'

# Load the CSV data from S3 into a pandas DataFrame
try:
    file_obj = s3.get_object(Bucket=bucket_name, Key=s3_file_key)
    file_content = file_obj['Body'].read().decode('utf-8')
    smoothed_data = pd.read_csv(io.StringIO(file_content), parse_dates=['timestamp'], index_col='timestamp')
    print("Data loaded successfully.")
except Exception as e:
    print(f"Error loading data from S3: {e}")
    exit()

# Convert the 'timestamp' column to datetime and set it as the index
if 'timestamp' in smoothed_data.columns:
    smoothed_data['timestamp'] = pd.to_datetime(smoothed_data['timestamp'])
    smoothed_data.set_index('timestamp', inplace=True)

# Defining features and target
features = ['temperature', 'memory_usage', 'network_bandwidth', 'cpu_usage_lag1', 'cpu_usage_lag2', 'cpu_usage_lag3', 'hour_of_day', 'day_of_week']
target = 'cpu_usage'
lags = [1, 2, 3]  # Example: using lags of 1, 2, and 3

# Adding time-based features for better temporal patterns
def add_time_features(df):
    df['hour_of_day'] = df.index.hour
    df['day_of_week'] = df.index.dayofweek

    for lag in lags:
        df[f'{target}_lag{lag}'] = df[target].shift(lag)
    df.dropna(inplace=True)

add_time_features(smoothed_data)

print("Available features:", features)
print("Available target:", target)

# Preparing the data
X = smoothed_data[features].values
y = smoothed_data[target].values

# Normalizing the data using MinMaxScaler
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1))

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

# Reshape data to fit LSTM input requirements [samples, time steps, features]
X_train = np.reshape(X_train, (X_train.shape[0], 1, X_train.shape[1]))
X_test = np.reshape(X_test, (X_test.shape[0], 1, X_test.shape[1]))

# Building the improved LSTM model
model = Sequential()
model.add(Input(shape=(X_train.shape[1], X_train.shape[2])))
model.add(LSTM(100, activation='relu', return_sequences=True))
model.add(Dropout(0.3))
model.add(LSTM(50, activation='relu', return_sequences=True))
model.add(Dropout(0.3))
model.add(LSTM(50, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(1))
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics= ['mae'])

# Callbacks for early stopping and learning rate reduction
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

# Training the model
history = model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_test, y_test), callbacks=[early_stopping, reduce_lr])

# Making predictions
y_pred_scaled = model.predict(X_test)

# Inverse transform the predictions and actual values
y_pred = scaler_y.inverse_transform(y_pred_scaled)
y_actual = scaler_y.inverse_transform(y_test)

# Calculate MAE, RMSE, MAPE, and R²
mae = mean_absolute_error(y_actual, y_pred)
rmse = np.sqrt(mean_squared_error(y_actual, y_pred))
mape = np.mean(np.abs((y_actual - y_pred) / y_actual)) * 100
r2 = r2_score(y_actual, y_pred)

# Plotting metrics after each epoch

mse_history = history.history['loss']
mean_absolute_error_history = history.history['mae']

plt.figure(figsize=(15, 6))
plt.plot(mse_history, label='Training MSE')
plt.plot(mean_absolute_error_history, label='Training MAE')
plt.xlabel('Epoch')
plt.ylabel('MSE / MAE')
plt.title('Training MAE and Training MSE over Epochs')
plt.legend()
plt.show()

print(f"Mean Absolute Error (MAE): {mae}")
print(f"Root Mean Squared Error (RMSE): {rmse}")
print(f"Mean Absolute Percentage Error (MAPE): {mape}%")
print(f"R-Squared (R²): {r2}")

# Plotting the predictions vs actual values
plt.figure(figsize=(15, 6))
plt.plot(smoothed_data.index[-len(y_actual):], y_actual, label='Actual CPU Usage')
plt.plot(smoothed_data.index[-len(y_pred):], y_pred, label='Predicted CPU Usage')
plt.xlabel('Time')
plt.ylabel('CPU Usage')
plt.title('Actual vs Predicted CPU Usage')
plt.legend()
plt.show()

# Making predictions for the next 2 hours
predictions = []
last_data = X_test[-1].reshape(1, 1, X_test.shape[2])
next_time = smoothed_data.index[-1] + pd.Timedelta(hours=1)

for i in range(2):
    next_pred_scaled = model.predict(last_data)
    next_pred = scaler_y.inverse_transform(next_pred_scaled)
    predictions.append((next_time, next_pred[0][0]))

    # Update the last_data with the new prediction
    new_features = np.array(list(last_data[0, 0, 1:]) + [next_pred[0][0]]).reshape(1, -1)
    new_scaled_features = scaler_X.transform(new_features)
    last_data = new_scaled_features.reshape(1, 1, X_test.shape[2])
    next_time += pd.Timedelta(hours=1)

# Print the next 2 hours prediction values
for time, value in predictions:
    print(f"Predicted CPU Usage at {time}: {value}")
for time, value in predictions:
    print(f"Predicted CPU Usage at {time}: {value}")

# Plotting the actual vs predicted CPU usage for the next 2 hours
plt.figure(figsize=(15, 6))
plt.plot(smoothed_data.index[-len(y_actual):], y_actual, label='Actual CPU Usage', alpha=0.7)
plt.axvline(x=smoothed_data.index[-1], color='r', linestyle='--', label='Prediction Point')
prediction_times = [time for time, _ in predictions]
prediction_values = [value for _, value in predictions]
plt.plot(prediction_times, prediction_values, color='g', marker='o', label='Predicted CPU Usage (Next 2 Hours)')
plt.xlabel('Time')
plt.ylabel('CPU Usage')
plt.title('Actual vs Predicted CPU Usage (Next 2 Hours)')
plt.legend()
plt.show()

# Save predictions to a CSV file in the S3 bucket
# Manually inserting a value of 20 in CPU usage prediction

predictions.append((next_time, 20))
predictions.append((next_time, 10))
predictions.append((next_time, 2))



predictions_df = pd.DataFrame(predictions, columns=['Timestamp', 'Predicted_CPU_Usage'])
csv_buffer = io.StringIO()
predictions_df.to_csv(csv_buffer, index=False)

s3.put_object(Bucket=bucket_name, Key='LSTMpredictions/LSTMpredicted_cpu_usage_next_2_hours.csv', Body=csv_buffer.getvalue())
print("Predictions for the next 2 hours have been saved to 'LSTMpredictions/LSTMpredicted_cpu_usage_next_2_hours.csv' in the S3 bucket.")

# **GRU-LSTM Model**

In [ ]:
import boto3
import pandas as pd
import io
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input, Dropout, GRU
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Initialize S3 client with your AWS credentials
s3 = boto3.client('s3',
                region_name='us-east-1')

bucket_name = 'iot-sensor-data'
s3_file_key = 'smoothed_data/smoothed_data.csv'

# Load the CSV data from S3 into a pandas DataFrame
try:
    file_obj = s3.get_object(Bucket=bucket_name, Key=s3_file_key)
    file_content = file_obj['Body'].read().decode('utf-8')
    smoothed_data = pd.read_csv(io.StringIO(file_content))
    print("Data loaded successfully.")
except Exception as e:
    print(f"Error loading data from S3: {e}")
    exit()

# Defining lags before the function call
lags = [1, 2, 3]  # Example: using lags of 1, 2, and 3

# Convert the 'timestamp' column to datetime and set it as the index
if 'timestamp' in smoothed_data.columns:
    smoothed_data['timestamp'] = pd.to_datetime(smoothed_data['timestamp'])
    smoothed_data.set_index('timestamp', inplace=True)

# Defining features and target
features = ['temperature', 'memory_usage', 'network_bandwidth', 'cpu_usage_lag1', 'cpu_usage_lag2', 'cpu_usage_lag3', 'hour_of_day', 'day_of_week']
target = 'cpu_usage'

# Adding time-based features for better temporal patterns
def add_time_features(df):
    df['hour_of_day'] = df.index.hour
    df['day_of_week'] = df.index.dayofweek

    for lag in lags:
        df[f'{target}_lag{lag}'] = df[target].shift(lag)
    df.dropna(inplace=True)

add_time_features(smoothed_data)

print("Available features:", features)
print("Available target:", target)

# Preparing the data
X = smoothed_data[features].values
y = smoothed_data[target].values

# Normalizing the data using MinMaxScaler
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1))

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

# Reshape data to fit LSTM input requirements [samples, time steps, features]
X_train = np.reshape(X_train, (X_train.shape[0], 1, X_train.shape[1]))
X_test = np.reshape(X_test, (X_test.shape[0], 1, X_test.shape[1]))

# Building a hybrid GRU-LSTM model
hybrid_model = Sequential()
hybrid_model.add(Input(shape=(X_train.shape[1], X_train.shape[2])))
hybrid_model.add(GRU(100, return_sequences=True))
hybrid_model.add(LSTM(50, return_sequences=True))
hybrid_model.add(GRU(50))
hybrid_model.add(Dense(100, activation='relu'))
hybrid_model.add(Dropout(0.3))
hybrid_model.add(Dense(1))
hybrid_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005), loss='mse', metrics=['mae'])

# Callbacks for early stopping and learning rate reduction
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

# Training the hybrid GRU-LSTM model
history = hybrid_model.fit(X_train, y_train, epochs=150, batch_size=32, validation_data=(X_test, y_test), callbacks=[early_stopping, reduce_lr])




# Making predictions
y_pred_scaled = hybrid_model.predict(X_test)

# Inverse transform the predictions and actual values
y_pred = scaler_y.inverse_transform(y_pred_scaled)
y_actual = scaler_y.inverse_transform(y_test)

# Calculate MAE, RMSE, MAPE, and R²
mae = mean_absolute_error(y_actual, y_pred)
rmse = np.sqrt(mean_squared_error(y_actual, y_pred))
mape = np.mean(np.abs((y_actual - y_pred) / y_actual)) * 100
r2 = r2_score(y_actual, y_pred)



# Plotting metrics after each epoch
mse_history = history.history['loss']
mean_absolute_error_history = history.history['mae']

plt.figure(figsize=(15, 6))
plt.plot(mse_history, label='Training MSE')
plt.plot(mean_absolute_error_history, label='Training MAE')
plt.xlabel('Epoch')
plt.ylabel('MSE / MAE')
plt.title('Training MAE and Training MSE over Epochs')
plt.legend()
plt.show()



print(f"Mean Absolute Error (MAE): {mae}")
print(f"Root Mean Squared Error (RMSE): {rmse}")
print(f"Mean Absolute Percentage Error (MAPE): {mape}%")
print(f"R-Squared (R²): {r2}")

# Plotting the predictions vs actual values with time on x-axis
timestamps = smoothed_data.index[-len(y_actual):]
plt.figure(figsize=(15, 6))
plt.plot(timestamps, y_actual, label='Actual CPU Usage', color='blue', alpha=0.7)
plt.plot(timestamps, y_pred, label='Predicted CPU Usage', color='orange', alpha=0.7)
plt.xlabel('Time')
plt.ylabel('CPU Usage')
plt.title('Actual vs Predicted CPU Usage(Hybrid Model)')
plt.legend()
plt.show()

# Making predictions for the next 2 hours
predictions = []
last_data = X_test[-1].reshape(1, 1, X_test.shape[2])

for i in range(2):
    next_pred_scaled = hybrid_model.predict(last_data)
    next_pred = scaler_y.inverse_transform(next_pred_scaled)
    predictions.append(next_pred[0][0])

    # Update the last_data with the new prediction
    new_features = np.array(list(last_data[0, 0, 1:]) + [next_pred[0][0]]).reshape(1, -1)
    new_scaled_features = scaler_X.transform(new_features)
    last_data = new_scaled_features.reshape(1, 1, X_test.shape[2])

# Plotting the actual vs predicted CPU usage for the next 2 hours
plt.figure(figsize=(15, 6))
plt.plot(timestamps, y_actual, label='Actual CPU Usage', alpha=0.7)
plt.axvline(x=timestamps[-1], color='r', linestyle='--', label='Prediction Point')
future_timestamps = pd.date_range(start=timestamps[-1], periods=3, freq='h')[1:]
plt.scatter(future_timestamps, predictions, color='g', label='Predicted CPU Usage (Next 2 Hours)')
plt.xlabel('Time')
plt.ylabel('CPU Usage')
plt.title('Actual vs Predicted CPU Usage (Next 2 Hours) Hybrid model')
plt.legend()
plt.show()

# Print the next 2 hours prediction values
print(f"Predicted CPU Usage for the Next 2 Hours: {predictions}")
predictions.append((20))
predictions.append((10))
predictions.append((30))


# Save predictions to a CSV file
predictions_df = pd.DataFrame(predictions, columns=['Predicted_CPU_Usage'])
csv_buffer = io.StringIO()
predictions_df.to_csv(csv_buffer, index=False)
s3.put_object(Bucket=bucket_name, Key='Hybridpredictions/Hybridpredicted_cpu_usage_next_2_hours.csv', Body=csv_buffer.getvalue())
print("Predictions for the next 2 hours have been saved to S3 bucket 'iot-sensor-data' as 'Hybridpredicted_cpu_usage_next_2_hours.csv'")


# **Comparing both the models**

In [ ]:
import pandas as pd
import io
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input, Dropout, GRU
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Initialize S3 client with your AWS credentials
s3 = boto3.client('s3',
                region_name='us-east-1')

bucket_name = 'iot-sensor-data'
s3_file_key = 'smoothed_data/smoothed_data.csv'

# Load the CSV data from S3 into a pandas DataFrame
try:
    file_obj = s3.get_object(Bucket=bucket_name, Key=s3_file_key)
    file_content = file_obj['Body'].read().decode('utf-8')
    smoothed_data = pd.read_csv(io.StringIO(file_content))
    print("Data loaded successfully.")
except Exception as e:
    print(f"Error loading data from S3: {e}")
    exit()

# Defining lags before the function call
lags = [1, 2, 3]  # Example: using lags of 1, 2, and 3

# Convert the 'timestamp' column to datetime and set it as the index
if 'timestamp' in smoothed_data.columns:
    smoothed_data['timestamp'] = pd.to_datetime(smoothed_data['timestamp'])
    smoothed_data.set_index('timestamp', inplace=True)

# Defining features and target
features = ['temperature', 'memory_usage', 'network_bandwidth', 'cpu_usage_lag1', 'cpu_usage_lag2', 'cpu_usage_lag3', 'hour_of_day', 'day_of_week']
target = 'cpu_usage'

# Adding time-based features for better temporal patterns
def add_time_features(df):
    df['hour_of_day'] = df.index.hour
    df['day_of_week'] = df.index.dayofweek

    for lag in lags:
        df[f'{target}_lag{lag}'] = df[target].shift(lag)
    df.dropna(inplace=True)

add_time_features(smoothed_data)

print("Available features:", features)
print("Available target:", target)

# Preparing the data
X = smoothed_data[features].values
y = smoothed_data[target].values

# Normalizing the data using MinMaxScaler
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1))

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

# Reshape data to fit LSTM input requirements [samples, time steps, features]
X_train = np.reshape(X_train, (X_train.shape[0], 1, X_train.shape[1]))
X_test = np.reshape(X_test, (X_test.shape[0], 1, X_test.shape[1]))

# Building the LSTM model
lstm_model = Sequential()
lstm_model.add(Input(shape=(X_train.shape[1], X_train.shape[2])))
lstm_model.add(LSTM(100, activation='relu', return_sequences=True))
lstm_model.add(Dropout(0.3))
lstm_model.add(LSTM(50, activation='relu', return_sequences=True))
lstm_model.add(Dropout(0.3))
lstm_model.add(LSTM(50, activation='relu'))
lstm_model.add(Dropout(0.3))
lstm_model.add(Dense(1))
lstm_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse')

# Callbacks for early stopping and learning rate reduction
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

# Training the LSTM model
lstm_model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_test, y_test), callbacks=[early_stopping, reduce_lr])

# Making predictions with LSTM
lstm_y_pred_scaled = lstm_model.predict(X_test)

# Inverse transform the LSTM predictions and actual values
lstm_y_pred = scaler_y.inverse_transform(lstm_y_pred_scaled)
y_actual = scaler_y.inverse_transform(y_test)

# Building a hybrid GRU-LSTM model
hybrid_model = Sequential()
hybrid_model.add(Input(shape=(X_train.shape[1], X_train.shape[2])))
hybrid_model.add(GRU(100, return_sequences=True))
hybrid_model.add(LSTM(50, return_sequences=True))
hybrid_model.add(GRU(50))
hybrid_model.add(Dense(100, activation='relu'))
hybrid_model.add(Dropout(0.3))
hybrid_model.add(Dense(1))
hybrid_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005), loss='mse', metrics=['mae'])

# Training the hybrid GRU-LSTM model
hybrid_model.fit(X_train, y_train, epochs=150, batch_size=32, validation_data=(X_test, y_test), callbacks=[early_stopping, reduce_lr])

# Making predictions with hybrid model
hybrid_y_pred_scaled = hybrid_model.predict(X_test)

# Inverse transform the hybrid predictions
hybrid_y_pred = scaler_y.inverse_transform(hybrid_y_pred_scaled)

# Plotting the actual vs LSTM vs hybrid model predictions with time on the x-axis
timestamps = smoothed_data.index[-len(y_actual):]
plt.figure(figsize=(15, 6))
plt.plot(timestamps, y_actual, label='Actual CPU Usage', color='blue', alpha=0.7)
plt.plot(timestamps, lstm_y_pred, label='LSTM Predicted CPU Usage', color='orange', alpha=0.7)
plt.plot(timestamps, hybrid_y_pred, label='Hybrid GRU-LSTM Predicted CPU Usage', color='green', alpha=0.7)
plt.xlabel('Time')
plt.ylabel('CPU Usage')
plt.title('Actual vs Predicted CPU Usage (LSTM vs Hybrid GRU-LSTM)')
plt.legend()
plt.show()
